## Configuration and Preprocessing

In [ ]:
import os
import numpy as np
from scipy.io import wavfile
from pydub import AudioSegment

# ============================================================
# CONFIGURATION
# ============================================================

FILES_DIR = "./files"

INPUT_MP3 = "raw.mp3"
HOST_SIGNAL_FILE = os.path.join(FILES_DIR, "preprocessed.wav")
WATERMARK_SIGNAL_FILE = os.path.join(FILES_DIR, "wmarked_file.wav")

PSEUDO_RAND_FILE = os.path.join(FILES_DIR, "pseudo_rand.dat")
WATERMARK_ORIGINAL_FILE = os.path.join(FILES_DIR, "watermark_binary.dat")
WATERMARK_EXTENDED_FILE = os.path.join(FILES_DIR, "watermark_extended.dat")

WATERMARK_TEXT = "Goweki"

REP_CODE = True
FRAME_LENGTH = 4096
CONTROL_STRENGTH = 0.1
OVERLAP = 0.0
NUM_REPS = 5

os.makedirs(FILES_DIR, exist_ok=True)

# ============================================================
# PREPROCESS AUDIO
# ============================================================

audio = AudioSegment.from_mp3(INPUT_MP3)
audio = audio.set_channels(1)
audio.export(HOST_SIGNAL_FILE, format="wav")

# ============================================================
# GENERATE PSEUDO-RANDOM SEQUENCE
# ============================================================

# prs = np.random.rand(FRAME_LENGTH) - 0.5

# with open(PSEUDO_RAND_FILE, "w") as f:
#     f.writelines(f"{value:.6f}\n" for value in prs)

np.random.seed(42)

prs = np.random.rand(FRAME_LENGTH) - 0.5

with open(PSEUDO_RAND_FILE, "w") as f:
    f.writelines(
        f"{value:.6f}\n"
        for value in prs
    )

# ============================================================
# LOAD HOST AUDIO
# ============================================================

sr, host_signal = wavfile.read(HOST_SIGNAL_FILE)

if host_signal.ndim > 1:
    host_signal = host_signal[:, 0]

host_signal = host_signal.astype(np.float64)

signal_len = len(host_signal)

frame_shift = int(
    FRAME_LENGTH * (1 - OVERLAP)
)

overlap_length = int(
    FRAME_LENGTH * OVERLAP
)

# Number of available embedding frames
embed_nbit = int(
    np.floor(
        (signal_len - overlap_length) / frame_shift
    )
)

# ============================================================
# CONVERT TEXT TO BINARY
# ============================================================

binary_str = "".join(
    format(ord(char), "08b")
    for char in WATERMARK_TEXT
)

if len(binary_str) > embed_nbit:
    binary_str = binary_str[:embed_nbit]
else:
    binary_str = binary_str.ljust(
        embed_nbit,
        "0"
    )

wmark_original = np.array(
    [int(bit) for bit in binary_str],
    dtype=np.int8
)

# ============================================================
# REPETITION CODING
# ============================================================

if REP_CODE:

    effective_nbit = embed_nbit // NUM_REPS

    embed_nbit = effective_nbit * NUM_REPS

    wmark_original = wmark_original[:effective_nbit]

    wmark_extended = np.repeat(
        wmark_original,
        NUM_REPS
    )

else:

    effective_nbit = embed_nbit
    wmark_extended = wmark_original

# Save generated watermark data
with open(WATERMARK_ORIGINAL_FILE, "w") as f:
    f.writelines(
        f"{bit}\n"
        for bit in wmark_original
    )

with open(WATERMARK_EXTENDED_FILE, "w") as f:
    f.writelines(
        f"{bit}\n"
        for bit in wmark_extended
    )

print("Preprocessing and watermark generation complete.")
print(f"Watermark text : {WATERMARK_TEXT}")
print(f"Watermark bits : {effective_nbit}")
print(f"PRS length     : {len(prs)}")

Watermark Embedding

In [ ]:
# ============================================================
# EMBEDDING
# ============================================================

pointer = 0

wmed_signal = np.zeros(
    frame_shift * embed_nbit
)

for i in range(embed_nbit):

    frame = host_signal[
        pointer:pointer + FRAME_LENGTH
    ].copy()

    # Embedding strength proportional to frame amplitude
    alpha = (
        CONTROL_STRENGTH *
        np.max(np.abs(frame))
    )

    # Binary phase modulation using the PRS
    if wmark_extended[i] == 1:
        frame += alpha * prs
    else:
        frame -= alpha * prs

    # Store the non-overlapping portion
    wmed_signal[
        frame_shift * i:
        frame_shift * (i + 1)
    ] = frame[:frame_shift]

    pointer += frame_shift

# Append remaining unmodified audio
if len(wmed_signal) < signal_len:

    wmed_signal = np.concatenate(
        (
            wmed_signal,
            host_signal[len(wmed_signal):signal_len]
        )
    )

# Prevent 16-bit overflow
wmed_signal = np.clip(
    wmed_signal,
    -32768,
    32767
).astype(np.int16)

wavfile.write(
    WATERMARK_SIGNAL_FILE,
    sr,
    wmed_signal
)

print("Watermark embedding complete.")
print(f"Output: {WATERMARK_SIGNAL_FILE}")

Detection, Decoding and Evaluation

In [ ]:
# ============================================================
# LOAD HOST AND WATERMARKED SIGNALS
# ============================================================

_, host_signal = wavfile.read(
    HOST_SIGNAL_FILE
)

_, eval_signal = wavfile.read(
    WATERMARK_SIGNAL_FILE
)

if host_signal.ndim > 1:
    host_signal = host_signal[:, 0]

if eval_signal.ndim > 1:
    eval_signal = eval_signal[:, 0]

host_signal = host_signal.astype(np.float64)
eval_signal = eval_signal.astype(np.float64)

signal_len = len(eval_signal)

frame_shift = int(
    FRAME_LENGTH * (1 - OVERLAP)
)

embed_nbit = int(
    np.floor(
        signal_len /
        frame_shift
    )
)

if REP_CODE:
    effective_nbit = embed_nbit // NUM_REPS
    embed_nbit = effective_nbit * NUM_REPS
else:
    effective_nbit = embed_nbit

# ============================================================
# LOAD REFERENCE WATERMARK AND PRS
# ============================================================

with open(WATERMARK_ORIGINAL_FILE, "r") as f:
    wmark_original = np.array(
        [int(x.strip()) for x in f if x.strip()],
        dtype=np.int8
    )

with open(PSEUDO_RAND_FILE, "r") as f:
    prs = np.array(
        [float(x.strip()) for x in f if x.strip()],
        dtype=np.float64
    )

# ============================================================
# DETECT WATERMARK BITS
# ============================================================

detected_bit = np.zeros(
    embed_nbit,
    dtype=np.int8
)

pointer = 0

for i in range(embed_nbit):

    # Difference between watermarked and original audio
    frame_diff = (
        eval_signal[
            pointer:pointer + FRAME_LENGTH
        ]
        -
        host_signal[
            pointer:pointer + FRAME_LENGTH
        ]
    )

    # Correlate difference signal with PRS
    correlation = np.correlate(
        frame_diff,
        prs,
        mode="full"
    )

    peak = np.argmax(
        np.abs(correlation)
    )

    detected_bit[i] = (
        1
        if correlation[peak] >= 0
        else 0
    )

    pointer += frame_shift

# ============================================================
# REPETITION DECODING
# ============================================================

if REP_CODE:

    wmark_recovered = np.zeros(
        effective_nbit,
        dtype=np.int8
    )

    for i in range(effective_nbit):

        start = i * NUM_REPS
        end = start + NUM_REPS

        chunk = detected_bit[start:end]

        wmark_recovered[i] = (
            1
            if np.mean(chunk) >= 0.5
            else 0
        )

else:

    wmark_recovered = detected_bit

# ============================================================
# DECODE BINARY TO TEXT
# ============================================================

recovered_binary = "".join(
    str(int(bit))
    for bit in wmark_recovered
)

chars = [
    recovered_binary[i:i + 8]
    for i in range(
        0,
        len(recovered_binary),
        8
    )
]

recovered_text = "".join(
    chr(int(char, 2))
    for char in chars
    if len(char) == 8
    and char != "00000000"
)

# ============================================================
# CALCULATE BER
# ============================================================

reference_bits = wmark_original[
    :len(wmark_recovered)
]

bit_errors = np.sum(
    reference_bits != wmark_recovered
)

BER = (
    bit_errors /
    len(reference_bits)
) * 100

# ============================================================
# CALCULATE SNR
# ============================================================

noise = (
    host_signal[:len(eval_signal)]
    -
    eval_signal
)

noise_power = np.sum(
    noise ** 2
)

signal_power = np.sum(
    host_signal[:len(eval_signal)] ** 2
)

SNR = (
    10 *
    np.log10(
        signal_power /
        noise_power
    )
)

# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 55)
print("WATERMARK DETECTION RESULTS")
print("=" * 55)

print(f"Original text   : {WATERMARK_TEXT}")
print(f"Recovered text  : {recovered_text}")
print(f"Watermark bits  : {len(reference_bits)}")
print(f"Bit errors      : {bit_errors}")
print(f"Bit Error Rate  : {BER:.2f}%")
print(f"Signal-to-Noise : {SNR:.2f} dB")

print("=" * 55)

## Blind Detection

In [ ]:
# ============================================================
# BLIND WATERMARK DETECTION
# Does not require the original host audio
# ============================================================

def detect_without_original():
    """
    Detect and recover the watermark directly from the
    watermarked audio without access to the original host signal.
    """

    # --------------------------------------------------------
    # 1. Load watermarked audio only
    # --------------------------------------------------------

    _, eval_signal = wavfile.read(
        WATERMARK_SIGNAL_FILE
    )

    if eval_signal.ndim > 1:
        eval_signal = eval_signal[:, 0]

    eval_signal = eval_signal.astype(np.float64)

    signal_len = len(eval_signal)

    frame_shift = int(
        FRAME_LENGTH * (1 - OVERLAP)
    )

    if frame_shift <= 0:
        raise ValueError(
            "FRAME_LENGTH and OVERLAP produce an invalid frame shift."
        )

    # Equivalent to the original fix() calculation.
    # With the current non-negative audio length, floor()
    # gives the required integer frame count.
    embed_nbit = int(
        np.floor(
            (
                signal_len
                - int(FRAME_LENGTH * OVERLAP)
            ) / frame_shift
        )
    )

    # --------------------------------------------------------
    # 2. Determine number of recoverable watermark bits
    # --------------------------------------------------------

    if REP_CODE:

        effective_nbit = int(
            np.floor(
                embed_nbit / NUM_REPS
            )
        )

        embed_nbit = (
            effective_nbit * NUM_REPS
        )

    else:

        effective_nbit = embed_nbit

    # --------------------------------------------------------
    # 3. Load PRS and original watermark for evaluation
    # --------------------------------------------------------

    with open(PSEUDO_RAND_FILE, "r") as f:
        prs = np.array(
            [
                float(x.strip())
                for x in f
                if x.strip()
            ],
            dtype=np.float64
        )

    # The original watermark is used only to calculate BER.
    # It is NOT used during blind detection.
    with open(WATERMARK_ORIGINAL_FILE, "r") as f:
        wmark_original = np.array(
            [
                int(x.strip())
                for x in f
                if x.strip()
            ],
            dtype=np.int8
        )

    if len(prs) != FRAME_LENGTH:
        raise ValueError(
            f"PRS length ({len(prs)}) does not match "
            f"FRAME_LENGTH ({FRAME_LENGTH})."
        )

    # --------------------------------------------------------
    # 4. Blind correlation detection
    # --------------------------------------------------------

    pointer = 0

    detected_bit = np.zeros(
        embed_nbit,
        dtype=np.int8
    )

    for i in range(embed_nbit):

        frame = eval_signal[
            pointer:pointer + FRAME_LENGTH
        ]

        if len(frame) < FRAME_LENGTH:
            break

        # Correlate the watermarked frame directly
        # with the pseudo-random sequence.
        correlation = np.correlate(
            frame,
            prs,
            mode="full"
        )

        peak = np.argmax(
            np.abs(correlation)
        )

        # Positive correlation peak = 1
        # Negative correlation peak = 0
        detected_bit[i] = (
            1
            if correlation[peak] >= 0
            else 0
        )

        pointer += frame_shift

    # --------------------------------------------------------
    # 5. Recover watermark using majority voting
    # --------------------------------------------------------

    if REP_CODE:

        wmark_recovered = np.zeros(
            effective_nbit,
            dtype=np.int8
        )

        for i in range(effective_nbit):

            start = i * NUM_REPS
            end = start + NUM_REPS

            chunk = detected_bit[
                start:end
            ]

            wmark_recovered[i] = (
                1
                if np.mean(chunk) >= 0.5
                else 0
            )

    else:

        wmark_recovered = detected_bit

    # --------------------------------------------------------
    # 6. Convert recovered binary to text
    # --------------------------------------------------------

    recovered_binary = "".join(
        str(int(bit))
        for bit in wmark_recovered
    )

    chars = [
        recovered_binary[i:i + 8]
        for i in range(
            0,
            len(recovered_binary),
            8
        )
    ]

    recovered_text = "".join(
        chr(int(char, 2))
        for char in chars
        if len(char) == 8
        and char != "00000000"
    )

    # --------------------------------------------------------
    # 7. Calculate BER
    # --------------------------------------------------------

    reference_bits = wmark_original[
        :len(wmark_recovered)
    ]

    if len(reference_bits) != len(wmark_recovered):
        raise ValueError(
            "Reference watermark and recovered watermark "
            "have different lengths."
        )

    bit_errors = np.sum(
        reference_bits != wmark_recovered
    )

    BER = (
        bit_errors /
        len(reference_bits)
    ) * 100

    # --------------------------------------------------------
    # 8. Display results
    # --------------------------------------------------------

    print("\n" + "=" * 55)
    print("BLIND WATERMARK DETECTION RESULTS")
    print("=" * 55)

    print(f"Original text  : {WATERMARK_TEXT}")
    print(f"Recovered text : {recovered_text}")
    print(f"Watermark bits : {len(reference_bits)}")
    print(f"Bit errors     : {bit_errors}")
    print(f"Bit Error Rate : {BER:.2f}%")

    print("=" * 55)

    return {
        "original_text": WATERMARK_TEXT,
        "recovered_text": recovered_text,
        "watermark_bits": len(reference_bits),
        "bit_errors": int(bit_errors),
        "BER": BER,
        "recovered_bits": wmark_recovered
    }


# ============================================================
# RUN BLIND DETECTION
# ============================================================

blind_results = detect_without_original()